## PyTorch 기초: 첫 딥러닝 모델 구축하기

이전 파트에서는 신경망의 기본 이론과 작동 원리에 대해 배웠습니다. 

이제는 이론을 코드로 옮겨 실제 딥러닝 모델을 만들어볼 차례입니다. 

이번 파트에서는 파이토치(PyTorch)의 핵심 구성 요소들을 배우고, 이를 활용하여 간단한 분류 모델을 직접 구현하는 과정을 안내합니다.

**이번 파트의 학습 목표:**

* PyTorch의 기본 데이터 구조인 `Tensor`를 이해하고 다룰 수 있습니다.
  
* 데이터를 모델에 효율적으로 공급하기 위한 `Dataset`과 `DataLoader`의 개념과 사용법을 익힙니다.
* PyTorch의 `nn.Module`을 상속받아 직접 신경망 모델의 구조를 정의할 수 있습니다.
* 이 모든 것을 종합하여 간단한 분류 모델의 전체 학습 과정을 경험합니다.

실습 프로젝트로는 `위스콘신 유방암 데이터셋(Wisconsin Breast Cancer)`을 사용하여, 종양의 특징 데이터를 기반으로 악성(malignant)과 양성(benign)을 분류하는 모델을 만들어 보겠습니다.

### 1. PyTorch의 심장, 텐서(Tensor)

텐서(Tensor)는 PyTorch의 모든 연산의 기본이 되는 핵심 데이터 구조입니다. NumPy의 `ndarray`와 매우 유사하지만, 두 가지 결정적인 차이점이 있습니다.

1.  `GPU 가속`: 텐서는 GPU를 사용하여 연산 속도를 크게 향상시킬 수 있습니다.
   
2.  `자동 미분`: 텐서는 `autograd`라는 자동 미분 엔진을 통해 신경망의 경사도(gradient)를 자동으로 계산해줍니다. 이는 역전파(backpropagation) 알고리즘을 손쉽게 구현할 수 있도록 돕습니다.

#### 1.1. 텐서 생성하기

다양한 방법으로 텐서를 생성할 수 있습니다.

In [2]:
# CPU 버전
!pip install torch torchvision torchaudio

  Using cached torch-2.7.1-cp311-cp311-win_amd64.whl.metadata (28 kB)
  Using cached torchvision-0.22.1-cp311-cp311-win_amd64.whl.metadata (6.1 kB)
  Using cached torchaudio-2.7.1-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached torch-2.7.1-cp311-cp311-win_amd64.whl (216.1 MB)
Using cached torchvision-0.22.1-cp311-cp311-win_amd64.whl (1.7 MB)
Using cached torchaudio-2.7.1-cp311-cp311-win_amd64.whl (2.5 MB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)

   ---------------------------------------- 0/5 [mpmath]
   ---------------------------------------- 0/5 [mpmath]
   ---------------------------------------- 0/5 [mpmath]
   ---------------------------------------- 0/5 [mpmath]
   ---------------------------------------- 0/5 [mpmath]
   ---------------------------------------- 0/5 [mpmath]
   -------

In [ ]:
# GPU 버전
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [4]:
import torch
import numpy as np

# Python 리스트로부터 생성
data_list = [[1, 2], [3, 4]]
x_data = torch.tensor(data_list)
print(f"리스트로부터 생성:\n {x_data} \n")

# NumPy 배열로부터 생성
np_array = np.array(data_list)
x_np = torch.from_numpy(np_array)
print(f"NumPy 배열로부터 생성:\n {x_np} \n")

# 다른 텐서의 속성(모양, 자료형)을 유지하며 생성
x_ones = torch.ones_like(x_data) # x_data와 동일한 shape과 dtype을 가지는 1로 채워진 텐서
print(f"ones_like:\n {x_ones} \n")

# rand_like : 평균은 0이고 표준편차는 1인 random으로 바꿔줌
x_rand = torch.rand_like(x_data, dtype=torch.float) # x_data의 shape을 따르되, dtype을 float으로 변경
print(f"rand_like:\n {x_rand} \n")

리스트로부터 생성:
 tensor([[1, 2],
        [3, 4]]) 

NumPy 배열로부터 생성:
 tensor([[1, 2],
        [3, 4]], dtype=torch.int32) 

ones_like:
 tensor([[1, 1],
        [1, 1]]) 

rand_like:
 tensor([[0.3496, 0.6794],
        [0.4156, 0.8773]]) 



#### 1.2. 텐서의 속성

텐서는 `shape`, `dtype`, 그리고 저장되는 `device`(CPU 또는 GPU)와 같은 속성을 가집니다.

In [5]:
tensor = torch.rand(3, 4)

print(f"Shape of tensor: {tensor.shape}")
print(f"Datatype of tensor: {tensor.dtype}")
print(f"Device tensor is stored on: {tensor.device}")

# GPU가 사용 가능하다면, 텐서를 GPU로 옮길 수 있습니다.
# 엔비디아에 있는 GPU를 쓸 수 있는 프로그래밍 모듈
if torch.cuda.is_available():
    tensor = tensor.to('cuda')
    print(f"Device tensor is stored on: {tensor.device}")

Shape of tensor: torch.Size([3, 4])
Datatype of tensor: torch.float32
Device tensor is stored on: cpu


### 2. 데이터를 모델에 공급하는 방법: `Dataset`과 `DataLoader`

딥러닝 모델을 학습시키려면, 대량의 데이터를 효율적으로 처리하고 모델에 공급하는 체계적인 방법이 필요합니다. PyTorch는 이를 위해 `Dataset`과 `DataLoader`라는 두 가지 강력한 도구를 제공합니다.

#### 2.1. `Dataset`: 데이터셋을 감싸는 표준화된 방법

`torch.utils.data.Dataset`은 데이터셋을 표현하는 추상 클래스입니다. 사용자 정의 데이터셋을 만들 때는 이 클래스를 상속받고, 다음 두 가지 메서드를 반드시 오버라이드(override)해야 합니다.

* `__len__()`: 데이터셋의 총 샘플 수를 반환합니다.
  
* `__getitem__(idx)`: 주어진 인덱스 `idx`에 해당하는 샘플 하나(일반적으로 데이터와 레이블의 쌍)를 반환합니다.

이 구조를 통해 어떤 데이터셋이든 동일한 방식으로 접근할 수 있게 됩니다.

#### 2.2. `DataLoader`: 배치 단위로 데이터를 손쉽게

`torch.utils.data.DataLoader`는 `Dataset`을 감싸서, 데이터를 미니배치(mini-batch) 형태로 제공하는 반복자(iterator)를 생성합니다. `DataLoader`의 주요 기능은 다음과 같습니다.

* **배치 처리 (`batch_size`):** 데이터를 지정된 크기의 묶음(배치)으로 만듭니다.
  
* **데이터 셔플링 (`shuffle`):** 매 에포크마다 데이터의 순서를 섞어 모델이 데이터의 순서에 과적합되는 것을 방지합니다.
* **병렬 처리 (`num_workers`):** 여러 개의 서브프로세스를 사용하여 데이터 로딩 속도를 높입니다.

간단한 예시를 통해 이 두 구성요소의 상호작용을 살펴봅시다.

In [6]:
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

# 1. 데이터 로드 및 분할
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. 사용자 정의 Dataset 클래스 생성
class BreastCancerDataset(Dataset):
    def __init__(self, features, labels):
        # 데이터를 float 텐서와 long 텐서로 변환하여 저장
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        # 데이터의 총 개수 반환
        return len(self.features)

    def __getitem__(self, idx):
        # 해당 인덱스의 데이터와 레이블을 반환
        return self.features[idx], self.labels[idx]

# 3. Dataset 인스턴스 생성
train_dataset = BreastCancerDataset(X_train, y_train)
test_dataset = BreastCancerDataset(X_test, y_test)

# 4. DataLoader 인스턴스 생성 - 데이터에 대한 하나의 타입이 됨
train_loader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=32, shuffle=False)

# DataLoader 사용 예시: 첫 번째 배치를 가져와 확인
features_batch, labels_batch = next(iter(train_loader)) # next - generator 의미미
print(f"Feature batch shape: {features_batch.size()}")
print(f"Labels batch shape: {labels_batch.size()}")

Feature batch shape: torch.Size([32, 30])
Labels batch shape: torch.Size([32])


### 3. PyTorch로 신경망 모델 정의하기

PyTorch에서 신경망 모델은 `torch.nn.Module` 클래스를 상속받아 정의합니다. 

`nn.Module`은 모델의 모든 계층(layer)과 `forward()` 메서드를 포함합니다.

모델을 정의할 때는 주로 다음 두 부분을 구현합니다.

* `__init__()`: 모델에 사용될 계층들(예: `nn.Linear`, `nn.ReLU` 등)을 초기화합니다.

* `forward(x)`: 입력 데이터 `x`가 모델의 각 계층을 통과하는 순서, 즉 데이터의 흐름을 정의합니다. PyTorch의 자동 미분 기능은 이 `forward` 연산을 추적하여 역전파 시 경사도를 계산합니다.

유방암 데이터 분류를 위한 간단한 다층 퍼셉트론(MLP) 모델을 만들어 보겠습니다. 

유방암 데이터셋은 30개의 특성을 가지므로, 입력층의 뉴런 수는 30이 됩니다. 출력은 '악성' 또는 '양성'의 2개 클래스이므로 출력층의 뉴런 수는 2가 됩니다.

In [7]:
import torch.nn as nn

class SimpleClassifier(nn.Module):
    def __init__(self, num_features, num_classes): # 특성수 30, 클래스수 2개 - 악성, 양성 분류
        super(SimpleClassifier, self).__init__()
        # 신경망 계층 정의
        # 데이터셋이 30개라 뉴런도 30으로 맞춤
        # linear 가중치의 제곱합 - 선형으로 연결
        # 1차 X / I(입력층) - 선형 - H1(은닉층1~16) / H2(은닉층2 8) - 출력2개와 매칭됨
        self.layer1 = nn.Linear(num_features, 16) # 입력 특성 30 -> 은닉층 16
        self.layer2 = nn.Linear(16, 8)            # 은닉층 16 -> 은닉층 8
        self.output_layer = nn.Linear(8, num_classes) # 은닉층 8 -> 출력 2

        self.relu = nn.ReLU() # 활성화 함수

    # 계산 그래프 생성
    def forward(self, x):
        # 데이터의 흐름 정의
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.output_layer(x)
        return x

# 모델 인스턴스 생성 및 구조 확인
input_features = X_train.shape[1]
output_classes = 2
model = SimpleClassifier(num_features=input_features, num_classes=output_classes)
print(model)

SimpleClassifier(
  (layer1): Linear(in_features=30, out_features=16, bias=True)
  (layer2): Linear(in_features=16, out_features=8, bias=True)
  (output_layer): Linear(in_features=8, out_features=2, bias=True)
  (relu): ReLU()
)


In [8]:
!pip install netron
!pip install onnx
!pip install torchsummary

  Using cached netron-8.4.0-py3-none-any.whl.metadata (1.5 kB)
Using cached netron-8.4.0-py3-none-any.whl (1.9 MB)


  Using cached onnx-1.18.0-cp311-cp311-win_amd64.whl.metadata (7.0 kB)
Using cached onnx-1.18.0-cp311-cp311-win_amd64.whl (15.8 MB)


  Using cached torchsummary-1.5.1-py3-none-any.whl.metadata (296 bytes)
Using cached torchsummary-1.5.1-py3-none-any.whl (2.8 kB)


In [16]:
# 더미 입력 생성 (배치 크기 1, 특성 30)
dummy_input = torch.randn(1, 30)

# ONNX 파일로 내보내기
torch.onnx.export(
    model, 
    dummy_input, 
    "models/torch/simple_classifier.onnx", 
    input_names=['input'], 
    output_names=['output'],
    opset_version=11
)
print("ONNX 파일로 저장 완료: simple_classifier.onnx")

ONNX 파일로 저장 완료: simple_classifier.onnx


In [18]:
!netron models/torch/simple_classifier.onnx

^C


In [13]:
from torchsummary import summary
# 모델 구조 요약 출력 (입력 크기 지정)
summary(model, input_size=(input_features,))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 16]             496
              ReLU-2                   [-1, 16]               0
            Linear-3                    [-1, 8]             136
              ReLU-4                    [-1, 8]               0
            Linear-5                    [-1, 2]              18
Total params: 650
Trainable params: 650
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00
----------------------------------------------------------------


### 4. 모델 학습의 전체 과정

이제 앞에서 배운 `DataLoader`, `Dataset`, 그리고 모델을 하나로 합쳐 전체 학습 과정을 완성해 보겠습니다.

학습 루프(training loop)는 일반적으로 다음과 같은 단계로 구성됩니다.

1.  **모델, 손실 함수, 옵티마이저 정의**: 모델 인스턴스를 생성하고, 학습 목표를 평가할 손실 함수(예: `nn.CrossEntropyLoss`)와 모델의 파라미터를 업데이트할 옵티마이저(예: `torch.optim.Adam`)를 정의합니다.
   
2.  **에포크(Epoch) 반복**: 전체 데이터셋을 여러 번 반복하여 학습합니다.
3.  **배치(Batch) 반복**: `DataLoader`를 통해 데이터를 배치 단위로 가져옵니다.
4.  **순전파(Forward Pass)**: 현재 배치를 모델에 입력하여 예측값을 계산합니다.
5.  **손실 계산**: 모델의 예측값과 실제 레이블을 비교하여 손실을 계산합니다.
6.  **역전파(Backward Pass)**: 손실에 대한 경사도를 계산(`loss.backward()`)합니다.
7.  **파라미터 업데이트**: 옵티마이저를 사용하여 모델의 가중치를 업데이트(`optimizer.step()`)합니다.
8.  **경사도 초기화**: 다음 배치를 위해 경사도를 0으로 초기화(`optimizer.zero_grad()`)합니다.

아래는 이 과정을 요약한 코드입니다.

In [16]:
import torch.optim as optim

# 1. 손실 함수와 옵티마이저 정의
criterion = nn.CrossEntropyLoss() # 이진분류 : 손실 함수를 계산산
optimizer = optim.Adam(model.parameters(), lr=0.001) # lr : 학습률 learning rate

# 2. 학습 루프
num_epochs = 20
for epoch in range(num_epochs):
    model.train() # 모델을 학습 모드로 설정

    running_loss = 0.0
    for features, labels in train_loader:
        # 8. 경사도 초기화
        optimizer.zero_grad()

        # 4. 순전파
        outputs = model(features) # model에 forward 함수 오버라이딩

        # 5. 손실 계산
        loss = criterion(outputs, labels)

        # 6. 역전파
        loss.backward()

        # 7. 파라미터 업데이트
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

print("\nFinished Training!")

Epoch [1/20], Loss: 2.1481
Epoch [2/20], Loss: 0.8406
Epoch [3/20], Loss: 0.6054
Epoch [4/20], Loss: 0.5167
Epoch [5/20], Loss: 0.4783
Epoch [6/20], Loss: 0.4176
Epoch [7/20], Loss: 0.3861
Epoch [8/20], Loss: 0.3582
Epoch [9/20], Loss: 0.3540
Epoch [10/20], Loss: 0.3177
Epoch [11/20], Loss: 0.3179
Epoch [12/20], Loss: 0.3044
Epoch [13/20], Loss: 0.3238
Epoch [14/20], Loss: 0.2962
Epoch [15/20], Loss: 0.2872
Epoch [16/20], Loss: 0.2787
Epoch [17/20], Loss: 0.2741
Epoch [18/20], Loss: 0.2718
Epoch [19/20], Loss: 0.2785
Epoch [20/20], Loss: 0.2776

Finished Training!
